# 04 - Mitigation: constant logit offset

Tests whether the directional bias is *additive*. If adding a single constant to
the Yes logit at inference recovers accuracy, the failure is a bias term rather
than degraded comprehension.

This is the causal test of finding 4: observing a shift and inferring it causes
the errors is correlational; removing the shift and recovering accuracy is not.

The crudeness is the point. A sophisticated fix working tells you less, because
sophisticated fixes work on many things.

Produces `results/mitigation/`.

> Reconstructed from session transcripts; outputs not embedded.

## Setup

Clone the repo, install deps, load Qwen2.5-VL-7B in bf16 across 2x T4.

**Check Accelerator = GPU T4 x2 before running.** A Kaggle batch job inherits
`None` silently and runs at ~1200 s/item on CPU. The assert below catches it.

In [ ]:
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GH_TOKEN")
import os, sys
if not os.path.isdir("/kaggle/working/algoverse"):
    !git clone https://{token}@github.com/bryantran21/algoverse.git /kaggle/working/algoverse
sys.path.insert(0, "/kaggle/working/algoverse"); os.chdir("/kaggle/working/algoverse")
!git config user.email "bryantran21@gmail.com"
!git config user.name  "bryantran21"

import subprocess
subprocess.run([sys.executable,'-m','pip','install','-q','transformers>=4.49.0',
    'accelerate>=0.34.0','datasets','qwen-vl-utils','typst','Pillow',
    'scikit-learn','matplotlib','tqdm'], check=True)

import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.device_count(), "devices")
assert torch.cuda.is_available(), "NO GPU - set Accelerator to T4 x2 before running"

import config
config.DEVICE_MAP = "auto"        # 2-GPU full precision
config.LOAD_IN_4BIT = False       # 4-bit perturbs activations - never use for interp
from src.inference import load_vl_model
model, processor = load_vl_model()
print("READY", flush=True)

## Collect Yes/No logits on both sets

Read `out.logits` directly rather than reconstructing through stored hidden
states - the two paths disagree (see nb 07) and this one is authoritative.

Both sets are needed: the control set gives the *independently measured* offset,
the failure set is where recovery is possible.

In [ ]:
import config, torch, gc, numpy as np, time, pickle, os
gc.collect(); torch.cuda.empty_cache()
os.makedirs("results/mitigation", exist_ok=True)

from src.inference import _messages_for, _flatten_images
YES, NO = 9454, 2753

@torch.no_grad()
def yes_no_logits(item, mode):
    msgs, images = _messages_for(item, mode)
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    imgs = _flatten_images([images])
    inputs = processor(text=[text], images=imgs if imgs else None,
                       return_tensors="pt").to(model.device)
    out = model(**inputs, use_cache=False)
    lg = out.logits[0, -1].float()
    return lg[YES].item(), lg[NO].item()

config.BATCH_SIZE = 2
config.RENDER["font_size_pt"] = 5.0

sets = pickle.load(open("results/sets/sets_n2000.pkl","rb"))
fail_all, ctrl_all = sets["fail_all"], sets["ctrl_all"]
print(f"{len(fail_all)} failure, {len(ctrl_all)} control", flush=True)

t0 = time.time()
for it in fail_all[:3]:
    yes_no_logits(it, "image")
print(f"{(time.time()-t0)/3:.2f} s/item", flush=True)   # sanity check before a long run

In [ ]:
def collect(items, tag):
    rows = []
    for k, it in enumerate(items):
        ty, tn  = yes_no_logits(it, "text")
        iy, inn = yes_no_logits(it, "image")
        rows.append(dict(item_id=it["item_id"], gold=it["gold"],
                         d_txt=ty-tn, d_img=iy-inn))
        if (k+1) % 50 == 0:
            pickle.dump(rows, open(f"results/mitigation/{tag}_logits.pkl","wb"))
            print(f"{tag} {k+1}/{len(items)}", flush=True)
    pickle.dump(rows, open(f"results/mitigation/{tag}_logits.pkl","wb"))
    return rows

fail_rows = collect(fail_all,       "fail")
ctrl_rows = collect(ctrl_all[:300], "ctrl")

for tag, rows in (("failure", fail_rows), ("control", ctrl_rows)):
    s = np.array([r["d_img"]-r["d_txt"] for r in rows])
    print(f"{tag}: shift {s.mean():+.3f}  sd {s.std():.3f}  "
          f"frac neg {(s<0).mean():.2f}  n={len(s)}", flush=True)

## Offset sweep

Sweep a constant added to the Yes logit and recompute accuracy offline - no extra
GPU per offset value.

Two numbers to report:
- **the measured offset** (|control shift|), predicted from activations and not
  fit to this evaluation set. This is the principled, non-cheating value.
- **the tuned optimum**, for reference only. It is fit on the same items it is
  evaluated on and should be labelled as such.

The evaluation union is deliberately failure-enriched, so the uncorrected
baseline is *not* the model's true image-mode accuracy. State this wherever the
number appears.

In [ ]:
import matplotlib.pyplot as plt

rows = fail_rows + ctrl_rows
gold = np.array([1 if r["gold"]=="yes" else 0 for r in rows])
d    = np.array([r["d_img"] for r in rows])
ctrl_shift = np.mean([r["d_img"]-r["d_txt"] for r in ctrl_rows])

def acc_at(off):
    p = (d + off) > 0
    return (p==gold).mean(), (p[gold==1]==1).mean(), (p[gold==0]==0).mean()

offsets = np.linspace(0, 5, 101)
res  = np.array([acc_at(o) for o in offsets])
best = int(res[:,0].argmax())
o_m  = abs(ctrl_shift)

a0,y0,n0 = acc_at(0.0)
a1,y1,n1 = acc_at(o_m)
aB,yB,nB = res[best]
print(f"n={len(rows)}  gold-yes {gold.sum()}  gold-no {(1-gold).sum()}")
print(f"uncorrected              acc {a0:.3f}  yes {y0:.3f}  no {n0:.3f}")
print(f"offset {o_m:.2f} (measured)   acc {a1:.3f}  yes {y1:.3f}  no {n1:.3f}")
print(f"tuned offset {offsets[best]:.2f}        acc {aB:.3f}  yes {yB:.3f}  no {nB:.3f}")

plt.figure(figsize=(7,4.5))
plt.plot(offsets, res[:,0], "-",  color="#1E2761", label="overall")
plt.plot(offsets, res[:,1], "--", color="#1E2761", label="gold = yes")
plt.plot(offsets, res[:,2], "--", color="#C0392B", label="gold = no")
plt.axvline(o_m, color="gray", ls=":", label=f"measured control shift ({o_m:.2f})")
plt.xlabel("logit offset added to Yes"); plt.ylabel("image-mode accuracy")
plt.title("Bias correction: constant offset at inference")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig("results/mitigation/offset_sweep.png", dpi=150); plt.show()

np.savez("results/mitigation/offset_sweep.npz",
         offsets=offsets, acc=res[:,0], acc_yes=res[:,1], acc_no=res[:,2],
         d=d, gold=gold, ctrl_shift=ctrl_shift)

In [ ]:
!git add -A && git commit -m "04: constant logit offset recovers image-mode accuracy" && git push origin master